In [67]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, confusion_matrix
from statsmodels.nonparametric.smoothers_lowess import lowess

Set constants and load data

In [68]:
# set variables
threshold = 0.3
look_back = 1
look_forward_time = 24

In [69]:
# set file path
# Make sure there are three folders: Code, Data, Results to read from and write to
file_path = "C:/Users/jenan/OneDrive - Tufts Medicine/code/EDSValidation_Jennie"
data_path = "C:/Users/jenan/OneDrive - Tufts Medicine/code/EDSValidation_Jennie/Data/Time_Dependent_All/No Refractory"
data_save_path = os.path.join(data_path, f'Threshold_{int(threshold * 100)}_Lookforward_{look_forward_time}_Lookback_{look_back}')
os.makedirs(data_save_path, exist_ok=True)

In [70]:
# # load no exclusion data
# result = pyreadr.read_r("C:/Users/jenan/OneDrive - Tufts Medicine/code/EDSValidation_Jennie/Data/no.exclusions.patient.data.RData")
# patient_data = result["patient.data"]
# print(patient_data.shape)

In [71]:
# load data
with open(f"{file_path}/Data/patient.data.pkl", 'rb') as f:
    data = pickle.load(f)
patient_data = data['patient_data']
print(patient_data.shape)

(4429852, 27)


Analysis

In [72]:
# create a list of dfs, 1 df per LCSN
patient_df_list = [group.reset_index(drop=True) for _, group in patient_data.groupby('LCSN')]
# remove dfs where patient doesn't have x hours of data
patient_df_list_filtered = [df for df in patient_df_list if len(df) >= (look_back * 4 + 1)]

In [73]:
# at each time point, look at time frame x hours back, look forward y hours to see outcome
# track max prediction in past x hours, and outcome in next y hours
# start from time point x * 4 (since we need to look back x hours)
# loop through list of dfs
new_patient_df_list = []
for patient_df in tqdm(patient_df_list_filtered):
    max_pred_list = []
    outcome_list = []

    for time_point in patient_df['time.point']:
        # start from time point x (since look back x hours), make sure sep outcome at the time is not 1
        if (time_point >= (look_back * 4)) and (patient_df['time.sep3.outcome'].iloc[time_point] != 1):
            max_pred = patient_df.iloc[(time_point - (look_back * 4)):time_point]['model.score'].max()
            outcome = (patient_df.iloc[time_point:time_point + (look_forward_time * 4)]['time.sep3.outcome'] == 1).any()
            max_pred_list.append(max_pred)
            outcome_list.append(outcome)
        else:
            # mark -1 if N/A
            max_pred_list.append(-1)
            outcome_list.append(-1)
    
    # create new columns of max model score in past x hours and outcome in x hours at each point
    new_df = patient_df.copy()
    new_df['look_back_max'] = max_pred_list
    new_df['look_forward_outcome'] = outcome_list
    new_patient_df_list.append(new_df)

100%|██████████| 31237/31237 [07:40<00:00, 67.77it/s] 


In [74]:
# recombine dfs, save
combined_df = pd.concat(new_patient_df_list, ignore_index=True)
# remove all rows where look_back_max or look_forward_outcome are -1 (only analyze on time points with valid look back max score and look forward outcome)
combined_df_filtered = combined_df[~((combined_df['look_back_max'] == -1) & (combined_df['look_forward_outcome'] == -1))]

In [75]:
# save
#combined_df_filtered.to_csv(os.path.join(data_save_path, 'Time_Dependent_Data.csv'), index = False)

GO FROM HERE FOR PLOT

In [76]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, confusion_matrix
from statsmodels.nonparametric.smoothers_lowess import lowess

In [77]:
# # set variables if not done above
# threshold = 0.25
# look_back = 2
# look_forward_time = 8

In [78]:
# # set file path if not done above
# # Make sure there are three folders: Code, Data, Results to read from and write to
# file_path = "C:/Users/jenan/OneDrive - Tufts Medicine/code/EDSValidation_Jennie"
# data_path = "C:/Users/jenan/OneDrive - Tufts Medicine/code/EDSValidation_Jennie/Data/Time_Dependent_All/No Refractory"
# data_save_path = os.path.join(data_path, f'Threshold_{int(threshold * 100)}_Lookforward_{look_forward_time}_Lookback_{look_back}')

In [79]:
# load data if not loaded above
# combined_df_filtered = pd.read_csv(os.path.join(data_save_path, 'Time_Dependent_Data.csv'))

In [80]:
# add new col for "prediction" if look_back_max is >= threshold of z
combined_df_filtered['time_dependent_prediction'] = (combined_df_filtered['look_back_max'] >= threshold).astype(int)

C:\Users\jenan\AppData\Local\Temp\ipykernel_20792\100695121.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_df_filtered['time_dependent_prediction'] = (combined_df_filtered['look_back_max'] >= threshold).astype(int)


In [81]:
# replace true in look_forward_outcome with 1 and false with 0
combined_df_filtered['look_forward_outcome'] = combined_df_filtered['look_forward_outcome'].astype(int)

C:\Users\jenan\AppData\Local\Temp\ipykernel_20792\1141360750.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_df_filtered['look_forward_outcome'] = combined_df_filtered['look_forward_outcome'].astype(int)


In [82]:
# create list of dfs grouped by time point
time_point_df_list = [group for _, group in combined_df_filtered.groupby('time.point')]

In [83]:
# # get metrics at each time point no bootstrapping
# # prediction = alarm went off (max score in past x hours is >= threshold)
# # outcome = if there was sepsis in the next 8 hours or not
# import warnings
# from sklearn.exceptions import UndefinedMetricWarning

# warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

# # loop through each time point
# time_point = []
# incidence = []
# tp = []
# fp = []
# tn = []
# fn = []
# auc = []
# sensitivity = []
# specificity = []
# ppv = []
# npv = []
# at_risk = []

# for df in tqdm(time_point_df_list):
#     predictions = df['time_dependent_prediction']
#     outcomes = df['look_forward_outcome']

#     time_point.append(df['time.point'].iloc[0])

#     incidence.append(np.mean(outcomes) * 100)
#     tp_val = np.sum((outcomes == 1) & (predictions == 1))  
#     fp_val = np.sum((outcomes == 0) & (predictions == 1))  
#     tn_val = np.sum((outcomes == 0) & (predictions == 0))  
#     fn_val = np.sum((outcomes == 1) & (predictions == 0))  
#     tp.append(tp_val)
#     fp.append(fp_val)
#     tn.append(tn_val)
#     fn.append(fn_val)

#     auc_value = roc_auc_score(outcomes, predictions)
#     auc.append(auc_value)

#     sensitivity_value = tp_val / (tp_val + fn_val) if tp_val + fn_val > 0 else 0
#     sensitivity.append(sensitivity_value)
#     specificity_value = tn_val / (tn_val + fp_val) if tn_val + fp_val > 0 else 0
#     specificity.append(specificity_value)

#     ppv_value = tp_val / (tp_val + fp_val) if tp_val + fp_val > 0 else 0
#     ppv.append(ppv_value)
#     npv_value = tn_val / (tn_val + fn_val) if tn_val + fn_val > 0 else 0
#     npv.append(npv_value)

#     at_risk.append(len(df))

In [84]:
# bootstrapping
import numpy as np
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from sklearn.exceptions import UndefinedMetricWarning
import warnings

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

n_boot = 500

time_point = []

# bootstrapped metrics
auc_boot = []
sens_boot = []
spec_boot = []
ppv_boot = []
npv_boot = []
incidence_boot = []

# NON-bootstrapped (point estimates only)
tp_list = []
fp_list = []
tn_list = []
fn_list = []
at_risk_list = []

for df in tqdm(time_point_df_list):

    predictions = df['time_dependent_prediction'].values
    outcomes = df['look_forward_outcome'].values

    time_point.append(df['time.point'].iloc[0])

    n = len(df)

    # -------------------------
    # store raw counts ONCE
    # -------------------------
    tp = np.sum((outcomes == 1) & (predictions == 1))
    fp = np.sum((outcomes == 0) & (predictions == 1))
    tn = np.sum((outcomes == 0) & (predictions == 0))
    fn = np.sum((outcomes == 1) & (predictions == 0))

    tp_list.append(tp)
    fp_list.append(fp)
    tn_list.append(tn)
    fn_list.append(fn)
    at_risk_list.append(n)

    # -------------------------
    # bootstrap metrics only
    # -------------------------
    auc_samples = []
    sens_samples = []
    spec_samples = []
    ppv_samples = []
    npv_samples = []
    inc_samples = []

    for _ in range(n_boot):

        idx = np.random.randint(0, n, n)

        y_pred = predictions[idx]
        y_true = outcomes[idx]

        # incidence (bootstrap OK)
        inc_samples.append(np.mean(y_true) * 100)

        # AUC
        try:
            auc_samples.append(roc_auc_score(y_true, y_pred))
        except ValueError:
            auc_samples.append(np.nan)

        # derived metrics
        tp_b = np.sum((y_true == 1) & (y_pred == 1))
        fp_b = np.sum((y_true == 0) & (y_pred == 1))
        tn_b = np.sum((y_true == 0) & (y_pred == 0))
        fn_b = np.sum((y_true == 1) & (y_pred == 0))

        sens_samples.append(tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else np.nan)
        spec_samples.append(tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else np.nan)
        ppv_samples.append(tp_b / (tp_b + fp_b) if (tp_b + fp_b) > 0 else np.nan)
        npv_samples.append(tn_b / (tn_b + fn_b) if (tn_b + fn_b) > 0 else np.nan)

    # store bootstrap distributions
    auc_boot.append(auc_samples)
    sens_boot.append(sens_samples)
    spec_boot.append(spec_samples)
    ppv_boot.append(ppv_samples)
    npv_boot.append(npv_samples)
    incidence_boot.append(inc_samples)

100%|██████████| 10142/10142 [26:59<00:00,  6.26it/s] 


In [85]:
def ci(x, low=2.5, high=97.5):
    x = np.array(x, dtype=float)
    return np.nanpercentile(x, low), np.nanpercentile(x, high)

summary = []

for i, tp in enumerate(time_point):

    auc_ci_low, auc_ci_high = ci(auc_boot[i])
    sens_ci_low, sens_ci_high = ci(sens_boot[i])
    spec_ci_low, spec_ci_high = ci(spec_boot[i])
    ppv_ci_low, ppv_ci_high = ci(ppv_boot[i])
    npv_ci_low, npv_ci_high = ci(npv_boot[i])
    inc_ci_low, inc_ci_high = ci(incidence_boot[i])

    summary.append({
        "Time Point": tp,

        # performance metrics
        "AUC": np.nanmean(auc_boot[i]),
        "AUC CI Low": auc_ci_low,
        "AUC CI High": auc_ci_high,

        "Sensitivity": np.nanmean(sens_boot[i]),
        "Sensitivity CI Low": sens_ci_low,
        "Sensitivity CI High": sens_ci_high,

        "Specificity": np.nanmean(spec_boot[i]),
        "Specificity CI Low": spec_ci_low,
        "Specificity CI High": spec_ci_high,

        "PPV": np.nanmean(ppv_boot[i]),
        "PPV CI Low": ppv_ci_low,
        "PPV CI High": ppv_ci_high,

        "NPV": np.nanmean(npv_boot[i]),
        "NPV CI Low": npv_ci_low,
        "NPV CI High": npv_ci_high,

        # incidence
        "Incidence": np.nanmean(incidence_boot[i]),
        "Incidence CI Low": inc_ci_low,
        "Incidence CI High": inc_ci_high,

        # deterministic counts (NO CI)
        "TP": tp_list[i],
        "FP": fp_list[i],
        "TN": tn_list[i],
        "FN": fn_list[i],

        # at risk (NO CI)
        "At Risk": at_risk_list[i],
    })

c:\Users\jenan\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1409: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
C:\Users\jenan\AppData\Local\Temp\ipykernel_20792\1243856042.py:20: RuntimeWarning: Mean of empty slice
  "AUC": np.nanmean(auc_boot[i]),
C:\Users\jenan\AppData\Local\Temp\ipykernel_20792\1243856042.py:24: RuntimeWarning: Mean of empty slice
  "Sensitivity": np.nanmean(sens_boot[i]),
C:\Users\jenan\AppData\Local\Temp\ipykernel_20792\1243856042.py:32: RuntimeWarning: Mean of empty slice
  "PPV": np.nanmean(ppv_boot[i]),


In [86]:
results_df = pd.DataFrame(summary)

In [87]:
# # get df of results
# results_df = pd.DataFrame({
#     'Time Point': time_point,
#     'Incidence': incidence,
#     'TP': tp,
#     'FP': fp,
#     'TN': tn,
#     'FN': fn,
#     'AUC': auc,
#     'Sensitivity': sensitivity,
#     'Specificity': specificity,
#     'PPV': ppv,
#     'NPV': npv,
#     'At Risk': at_risk
# })

In [88]:
# reorder results df
results_df.sort_values(by='Time Point', ascending=True, inplace=False)

,Time Point,AUC,AUC CI Low,AUC CI High,Sensitivity,Sensitivity CI Low,Sensitivity CI High,Specificity,Specificity CI Low,Specificity CI High,...,NPV CI Low,NPV CI High,Incidence,Incidence CI Low,Incidence CI High,TP,FP,TN,FN,At Risk
0,4,0.698804,0.678054,0.719292,0.478351,0.436608,0.519617,0.919257,0.916454,0.922250,...,0.988561,0.990933,1.790165,1.636348,1.929735,267,2475,28162,292,31196
1,5,0.686340,0.666207,0.708142,0.424798,0.384284,0.468698,0.947883,0.945395,0.950430,...,0.988281,0.990667,1.728871,1.588770,1.872163,221,1544,28115,299,30179
2,6,0.675391,0.653763,0.697239,0.393694,0.350313,0.437504,0.957087,0.954557,0.959148,...,0.988210,0.990633,1.652003,1.513917,1.791484,189,1225,27325,290,29029
3,7,0.665595,0.644092,0.687729,0.362878,0.319824,0.406842,0.968311,0.966300,0.970282,...,0.988138,0.990674,1.603124,1.441491,1.747624,161,868,26469,284,27782
4,8,0.659714,0.638772,0.682859,0.348325,0.306487,0.394510,0.971102,0.969217,0.973107,...,0.987985,0.990370,1.594135,1.445392,1.732206,147,754,25321,276,26498
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10137,10141,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0,0,1,0,1
10138,10142,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0,0,1,0,1
10139,10143,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0,0,1,0,1
10140,10144,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0,0,1,0,1


In [89]:
# save
#results_df.to_csv(os.path.join(data_save_path, 'Time_dependent_metric_results.csv'), index = False)

In [90]:
# subset df for 4 days, set legend to be in days
subset_results_df = results_df[results_df['Time Point'] <= 336]
subset_results_df['Time Point Days'] = subset_results_df['Time Point'] / 96

C:\Users\jenan\AppData\Local\Temp\ipykernel_20792\2372630985.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset_results_df['Time Point Days'] = subset_results_df['Time Point'] / 96


In [91]:
days = [8, 48, 96, 144, 192, 240, 288, 336]
table_df = subset_results_df[subset_results_df['Time Point'].isin(days)][['Time Point Days', 'Incidence', 'At Risk', 'TP', 'FP', 'TN', 'FN']]
table_df[['Time Point Days', 'Incidence', 'At Risk', 'TP', 'FP', 'TN', 'FN']] = table_df[['Time Point Days', 'Incidence', 'At Risk', 'TP', 'FP', 'TN', 'FN']].round(2)
table_df = table_df.T

In [92]:
def check_nans_in_columns(df, columns):
    for col in columns:
        n_missing = df[col].isna().sum()
        print(f"Column '{col}' has {n_missing} missing values (NaNs)")
cols_to_check = ['Time Point Days', 'Incidence', 'AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV']
check_nans_in_columns(subset_results_df, cols_to_check)

Column 'Time Point Days' has 0 missing values (NaNs)
Column 'Incidence' has 0 missing values (NaNs)
Column 'AUC' has 0 missing values (NaNs)
Column 'Sensitivity' has 0 missing values (NaNs)
Column 'Specificity' has 0 missing values (NaNs)
Column 'PPV' has 0 missing values (NaNs)
Column 'NPV' has 0 missing values (NaNs)


In [93]:
subset_results_df['AUC'] = subset_results_df['AUC'].fillna(method='ffill')

C:\Users\jenan\AppData\Local\Temp\ipykernel_20792\724104491.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  subset_results_df['AUC'] = subset_results_df['AUC'].fillna(method='ffill')
C:\Users\jenan\AppData\Local\Temp\ipykernel_20792\724104491.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset_results_df['AUC'] = subset_results_df['AUC'].fillna(method='ffill')


In [94]:
subset_results_df_to_smooth = subset_results_df.copy()

In [95]:
# LOWESS SMOOTHING FOR EACH COL BASED ON TIME POINT DAYS X
def apply_lowess_multiple(df, x_col, y_cols, frac):
    for col in y_cols:
        smoothed = lowess(df[col], df[x_col], frac)
        df[col] = smoothed[:, 1]
    return df
subset_results_df_smoothed = apply_lowess_multiple(subset_results_df_to_smooth, 'Time Point Days', ['Incidence', 'AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV'], 0.15)

In [96]:
# import matplotlib.pyplot as plt
# from matplotlib.gridspec import GridSpec
# import os

# # Base font sizes
# label_fontsize = 16
# title_fontsize = 18
# tick_fontsize = 14
# table_fontsize = 16

# # Figure size: wider to match table
# fig = plt.figure(figsize=(16, 20))

# # GridSpec: 4 rows, 2 columns
# # height_ratios: 3 rows of plots, 1 row for table (taller)
# # width_ratios: both columns equal width
# gs = GridSpec(4, 2, figure=fig, height_ratios=[1, 1, 1, 1.5], width_ratios=[1, 1], hspace=0.4, wspace=0.3)

# # ---- Create axes ----
# # Left column plots
# ax_inc = fig.add_subplot(gs[0, 0])
# ax_sens = fig.add_subplot(gs[1, 0], sharex=ax_inc)
# ax_spec = fig.add_subplot(gs[2, 0], sharex=ax_inc)

# # Right column plots
# ax_ppv = fig.add_subplot(gs[0, 1], sharex=ax_inc)
# ax_npv = fig.add_subplot(gs[1, 1], sharex=ax_inc)
# ax_auc = fig.add_subplot(gs[2, 1], sharex=ax_inc)

# # Bottom table spanning both columns
# ax_table = fig.add_subplot(gs[3, :])

# # ---- Left column ----
# ax_inc.plot(subset_results_df_smoothed['Time Point Days'],
#             subset_results_df_smoothed['Incidence'])
# ax_inc.set_ylabel('Incidence', fontsize=label_fontsize)
# weighted_avg = (subset_results_df_smoothed['Incidence'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_inc.set_title(f'Incidence (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_inc.tick_params(axis='both', labelsize=tick_fontsize)

# ax_sens.plot(subset_results_df_smoothed['Time Point Days'],
#              subset_results_df_smoothed['Sensitivity'])
# ax_sens.set_ylabel('Sensitivity', fontsize=label_fontsize)
# weighted_avg = (subset_results_df_smoothed['Sensitivity'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_sens.set_title(f'Sensitivity (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_sens.tick_params(axis='both', labelsize=tick_fontsize)

# ax_spec.plot(subset_results_df_smoothed['Time Point Days'],
#              subset_results_df_smoothed['Specificity'])
# ax_spec.set_ylabel('Specificity', fontsize=label_fontsize)
# weighted_avg = (subset_results_df_smoothed['Specificity'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_spec.set_title(f'Specificity (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_spec.tick_params(axis='both', labelsize=tick_fontsize)

# # ---- Right column ----
# ax_ppv.plot(subset_results_df_smoothed['Time Point Days'],
#             subset_results_df_smoothed['PPV'])
# ax_ppv.set_ylabel('PPV', fontsize=label_fontsize)
# ax_ppv.set_ylim(0.0, 0.3)
# weighted_avg = (subset_results_df_smoothed['PPV'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_ppv.set_title(f'PPV (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_ppv.tick_params(axis='both', labelsize=tick_fontsize)

# ax_npv.plot(subset_results_df_smoothed['Time Point Days'],
#             subset_results_df_smoothed['NPV'])
# ax_npv.set_ylabel('NPV', fontsize=label_fontsize)
# ax_npv.set_ylim(0.5, 1.0)
# weighted_avg = (subset_results_df_smoothed['NPV'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_npv.set_title(f'NPV (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_npv.tick_params(axis='both', labelsize=tick_fontsize)

# ax_auc.plot(subset_results_df_smoothed['Time Point Days'],
#             subset_results_df_smoothed['AUC'])
# ax_auc.set_ylabel('AUC', fontsize=label_fontsize)
# ax_auc.set_ylim(0.5, 1.0)
# q1 = subset_results_df_smoothed['AUC'].quantile(0.25)
# q3 = subset_results_df_smoothed['AUC'].quantile(0.75)
# weighted_avg = (subset_results_df_smoothed['AUC'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()
# ax_auc.set_title(f'AUC (weighted avg: {round(weighted_avg, 2)}, IQR: [{round(q1, 2)}, {round(q3, 2)}])', fontsize=title_fontsize)
# ax_auc.tick_params(axis='both', labelsize=tick_fontsize)

# # Shared x-label for bottom plots
# ax_spec.set_xlabel('Time Point (Days)', fontsize=label_fontsize)
# ax_auc.set_xlabel('Time Point (Days)', fontsize=label_fontsize)

# # ---- Bottom table ----
# ax_table.axis("off")
# table = ax_table.table(
#     cellText=table_df.values,
#     rowLabels=['Time Point Days', 'Incidence', 'At Risk', 'TP', 'FP', 'TN', 'FN'],
#     loc="center",
#     cellLoc='center',
#     rowLoc='center',
#     colWidths=[0.15] * table_df.shape[1]
# )

# # Table font and row height
# table.auto_set_font_size(False)
# table.set_fontsize(table_fontsize)
# for key, cell in table.get_celld().items():
#     cell.set_height(0.09)  # increase row height
#     cell.set_linewidth(1.5)  # thicker grid lines

# # Adjust layout
# plt.tight_layout()
# plt.savefig(os.path.join(data_save_path, 'metric_figure_smoothed.png'), bbox_inches="tight")
# plt.show()

In [97]:
# import matplotlib.pyplot as plt
# from matplotlib.gridspec import GridSpec
# import os

# # Base font sizes
# label_fontsize = 16
# title_fontsize = 18
# tick_fontsize = 14
# table_fontsize = 16

# fig = plt.figure(figsize=(16, 20))

# gs = GridSpec(
#     4, 2,
#     figure=fig,
#     height_ratios=[1, 1, 1, 1.5],
#     width_ratios=[1, 1],
#     hspace=0.4,
#     wspace=0.3
# )

# # ---- Axes ----
# ax_inc = fig.add_subplot(gs[0, 0])
# ax_sens = fig.add_subplot(gs[1, 0], sharex=ax_inc)
# ax_spec = fig.add_subplot(gs[2, 0], sharex=ax_inc)

# ax_ppv = fig.add_subplot(gs[0, 1], sharex=ax_inc)
# ax_npv = fig.add_subplot(gs[1, 1], sharex=ax_inc)
# ax_auc = fig.add_subplot(gs[2, 1], sharex=ax_inc)

# ax_table = fig.add_subplot(gs[3, :])

# x = subset_results_df_smoothed['Time Point Days']

# # =========================
# # INCIDENCE
# # =========================
# ax_inc.plot(x, subset_results_df_smoothed['Incidence'], color='C0')
# ax_inc.fill_between(
#     x,
#     subset_results_df_smoothed['Incidence CI Low'],
#     subset_results_df_smoothed['Incidence CI High'],
#     alpha=0.2,
#     color='C0'
# )

# weighted_avg = (subset_results_df_smoothed['Incidence'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

# ax_inc.set_ylabel('Incidence', fontsize=label_fontsize)
# ax_inc.set_title(f'Incidence (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_inc.tick_params(axis='both', labelsize=tick_fontsize)

# # =========================
# # SENSITIVITY
# # =========================
# ax_sens.plot(x, subset_results_df_smoothed['Sensitivity'], color='C1')
# ax_sens.fill_between(
#     x,
#     subset_results_df_smoothed['Sensitivity CI Low'],
#     subset_results_df_smoothed['Sensitivity CI High'],
#     alpha=0.2,
#     color='C1'
# )

# weighted_avg = (subset_results_df_smoothed['Sensitivity'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

# ax_sens.set_ylabel('Sensitivity', fontsize=label_fontsize)
# ax_sens.set_title(f'Sensitivity (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_sens.tick_params(axis='both', labelsize=tick_fontsize)

# # =========================
# # SPECIFICITY
# # =========================
# ax_spec.plot(x, subset_results_df_smoothed['Specificity'], color='C2')
# ax_spec.fill_between(
#     x,
#     subset_results_df_smoothed['Specificity CI Low'],
#     subset_results_df_smoothed['Specificity CI High'],
#     alpha=0.2,
#     color='C2'
# )

# weighted_avg = (subset_results_df_smoothed['Specificity'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

# ax_spec.set_ylabel('Specificity', fontsize=label_fontsize)
# ax_spec.set_xlabel('Time Point (Days)', fontsize=label_fontsize)
# ax_spec.set_title(f'Specificity (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_spec.tick_params(axis='both', labelsize=tick_fontsize)

# # =========================
# # PPV
# # =========================
# ax_ppv.plot(x, subset_results_df_smoothed['PPV'], color='C3')
# ax_ppv.fill_between(
#     x,
#     subset_results_df_smoothed['PPV CI Low'],
#     subset_results_df_smoothed['PPV CI High'],
#     alpha=0.2,
#     color='C3'
# )

# weighted_avg = (subset_results_df_smoothed['PPV'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

# ax_ppv.set_ylabel('PPV', fontsize=label_fontsize)
# ax_ppv.set_ylim(0.0, 0.3)
# ax_ppv.set_title(f'PPV (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_ppv.tick_params(axis='both', labelsize=tick_fontsize)

# # =========================
# # NPV
# # =========================
# ax_npv.plot(x, subset_results_df_smoothed['NPV'], color='C4')
# ax_npv.fill_between(
#     x,
#     subset_results_df_smoothed['NPV CI Low'],
#     subset_results_df_smoothed['NPV CI High'],
#     alpha=0.2,
#     color='C4'
# )

# weighted_avg = (subset_results_df_smoothed['NPV'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

# ax_npv.set_ylabel('NPV', fontsize=label_fontsize)
# ax_npv.set_ylim(0.5, 1.0)
# ax_npv.set_title(f'NPV (weighted avg: {round(weighted_avg, 2)})', fontsize=title_fontsize)
# ax_npv.tick_params(axis='both', labelsize=tick_fontsize)

# # =========================
# # AUC
# # =========================
# ax_auc.plot(x, subset_results_df_smoothed['AUC'], color='C5')
# ax_auc.fill_between(
#     x,
#     subset_results_df_smoothed['AUC CI Low'],
#     subset_results_df_smoothed['AUC CI High'],
#     alpha=0.2,
#     color='C5'
# )

# q1 = subset_results_df_smoothed['AUC'].quantile(0.25)
# q3 = subset_results_df_smoothed['AUC'].quantile(0.75)

# weighted_avg = (subset_results_df_smoothed['AUC'] * subset_results_df_smoothed['At Risk']).sum() / subset_results_df_smoothed['At Risk'].sum()

# ax_auc.set_ylabel('AUC', fontsize=label_fontsize)
# ax_auc.set_ylim(0.5, 1.0)
# ax_auc.set_xlabel('Time Point (Days)', fontsize=label_fontsize)
# ax_auc.set_title(
#     f'AUC (weighted avg: {round(weighted_avg, 2)}, IQR: [{round(q1, 2)}, {round(q3, 2)}])',
#     fontsize=title_fontsize
# )
# ax_auc.tick_params(axis='both', labelsize=tick_fontsize)

# # =========================
# # TABLE
# # =========================
# ax_table.axis("off")

# table = ax_table.table(
#     cellText=table_df.values,
#     rowLabels=['Time Point Days', 'Incidence', 'At Risk', 'TP', 'FP', 'TN', 'FN'],
#     loc="center",
#     cellLoc='center',
#     rowLoc='center',
#     colWidths=[0.15] * table_df.shape[1]
# )

# table.auto_set_font_size(False)
# table.set_fontsize(table_fontsize)

# for key, cell in table.get_celld().items():
#     cell.set_height(0.09)
#     cell.set_linewidth(1.5)

# # =========================
# # SAVE
# # =========================
# plt.tight_layout()
# plt.savefig(
#     os.path.join(data_save_path, 'metric_figure_smoothed.png'),
#     bbox_inches="tight"
# )
# plt.show()

In [98]:
# table_df.to_csv(os.path.join(data_save_path, 'table.csv'))

In [99]:
df = subset_results_df_smoothed

# -------------------------
# Helpers
# -------------------------
def weighted_avg(x, w):
    return np.sum(x * w) / np.sum(w)

def iqr(x):
    return np.nanpercentile(x, 75) - np.nanpercentile(x, 25)

# -------------------------
# Metrics
# -------------------------
metrics = {
    "Incidence": df["Incidence"],
    "Sensitivity": df["Sensitivity"],
    "Specificity": df["Specificity"],
    "PPV": df["PPV"],
    "NPV": df["NPV"],
    "AUC": df["AUC"],
}

weights = df["At Risk"]

summary_rows = []

for name, series in metrics.items():
    summary_rows.append({
        "Metric": name,
        "Weighted Mean": weighted_avg(series.values, weights.values),
        "Mean": np.nanmean(series),
        "Median": np.nanmedian(series),
        "IQR": iqr(series),
        "Q1": np.nanpercentile(series, 25),
        "Q3": np.nanpercentile(series, 75),
        "Min": np.nanmin(series),
        "Max": np.nanmax(series),
    })

summary_df = pd.DataFrame(summary_rows)

# -------------------------
# Save CSV
# -------------------------
output_path = os.path.join(data_save_path, "metric_summary.csv")
summary_df.to_csv(output_path, index=False)